In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("agro.db")
datos = pd.read_sql_query("""
    SELECT
        e.cultivo,
        e.provincia,
        e.anio,
        SUM(e.produccion_tm) * 1000.0 / SUM(e.superficie_cosechada_ha) AS rendimiento,
        c.precip_total_mm, c.temp_media, c.temp_max_media,
        c.radiacion_solar, c.humedad_rel, c.viento, c.humedad_suelo
    FROM estimaciones e
    JOIN clima c ON e.provincia = c.provincia AND e.anio = c.anio
    WHERE e.cultivo IN ('soja total', 'maíz', 'trigo total', 'girasol', 'sorgo')
      AND e.superficie_cosechada_ha > 0
      AND e.produccion_tm > 0
    GROUP BY e.cultivo, e.provincia, e.anio
""", conn)
conn.close()

print("Forma:", datos.shape)
datos.head()

Forma: (1421, 11)


,cultivo,provincia,anio,rendimiento,precip_total_mm,temp_media,temp_max_media,radiacion_solar,humedad_rel,viento,humedad_suelo
0,girasol,Buenos Aires,2000,1560.922867,1148.693333,14.804672,21.353434,16.926521,72.645055,2.794253,0.594335
1,girasol,Buenos Aires,2001,1746.583927,1380.553333,15.254621,21.434922,16.461215,76.496594,2.889288,0.634274
2,girasol,Buenos Aires,2002,1496.359806,1202.000000,14.870868,21.277498,16.723297,74.780959,2.864064,0.629059
3,girasol,Buenos Aires,2003,1704.487245,824.900000,15.334685,22.268922,17.519434,68.947489,2.957233,0.564776
4,girasol,Buenos Aires,2004,1940.213725,842.913333,15.866175,22.663133,17.096849,68.421840,2.936603,0.540592


In [2]:
#preparamos datos para sklearn
from sklearn.model_selection import train_test_split

y = datos["rendimiento"]
X = datos.drop(columns=["rendimiento"])

X = pd.get_dummies(X, columns=["cultivo", "provincia"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Entrenamiento:", X_train.shape, "Prueba:", X_test.shape)
X.head()

Entrenamiento: (1136, 26) Prueba: (285, 26)


,anio,precip_total_mm,temp_media,temp_max_media,radiacion_solar,humedad_rel,viento,humedad_suelo,cultivo_girasol,cultivo_maíz,...,provincia_Córdoba,provincia_Entre Ríos,provincia_Jujuy,provincia_La Pampa,provincia_Misiones,provincia_Salta,provincia_San Luis,provincia_Santa Fe,provincia_Santiago del Estero,provincia_Tucumán
0,2000,1148.693333,14.804672,21.353434,16.926521,72.645055,2.794253,0.594335,True,False,...,False,False,False,False,False,False,False,False,False,False
1,2001,1380.553333,15.254621,21.434922,16.461215,76.496594,2.889288,0.634274,True,False,...,False,False,False,False,False,False,False,False,False,False
2,2002,1202.000000,14.870868,21.277498,16.723297,74.780959,2.864064,0.629059,True,False,...,False,False,False,False,False,False,False,False,False,False
3,2003,824.900000,15.334685,22.268922,17.519434,68.947489,2.957233,0.564776,True,False,...,False,False,False,False,False,False,False,False,False,False
4,2004,842.913333,15.866175,22.663133,17.096849,68.421840,2.936603,0.540592,True,False,...,False,False,False,False,False,False,False,False,False,False


In [4]:
#Entrenamos y evaluamos

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

modelo = RandomForestRegressor(n_estimators=200, random_state=42)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)

print("R2:", round(r2_score(y_test, pred), 3))
print("Error medio (MAE):", round(mean_absolute_error(y_test, pred)), "kg/ha")

R2: 0.845
Error medio (MAE): 526 kg/ha


In [ ]:
#Que variables climaticas se relaciona mas con el rinde?
importancias = pd.DataFrame({
    "variable": X.columns,
    "importancia": modelo.feature_importances_
}).sort_values("importancia", ascending=False)

importancias.head(15)

,variable,importancia
9,cultivo_maíz,0.400507
11,cultivo_sorgo,0.145547
2,temp_media,0.116069
6,viento,0.065350
0,anio,0.065063
7,humedad_suelo,0.048153
1,precip_total_mm,0.034312
3,temp_max_media,0.029910
5,humedad_rel,0.026845
4,radiacion_solar,0.020143


In [ ]:
#Ya que el modelo se quedo mirando como diferenciar los cultivos por sus distintos rindes vamos con un solo cultivo
soja = datos[datos["cultivo"] == "soja total"].copy()

y_s = soja["rendimiento"]
X_s = soja.drop(columns=["rendimiento", "cultivo"])
X_s = pd.get_dummies(X_s, columns=["provincia"])

X_tr, X_te, y_tr, y_te = train_test_split(X_s, y_s, test_size=0.2, random_state=42)

modelo_s = RandomForestRegressor(n_estimators=200, random_state=42)
modelo_s.fit(X_tr, y_tr)
pred_s = modelo_s.predict(X_te)

print("R²:", round(r2_score(y_te, pred_s), 3))
print("MAE:", round(mean_absolute_error(y_te, pred_s)), "kg/ha")

imp_s = (pd.DataFrame({"variable": X_s.columns, "importancia": modelo_s.feature_importances_})
         .sort_values("importancia", ascending=False))
imp_s.head(12)

R²: 0.308
MAE: 392 kg/ha


,variable,importancia
0,anio,0.165849
6,viento,0.135890
4,radiacion_solar,0.135371
7,humedad_suelo,0.111540
1,precip_total_mm,0.087204
5,humedad_rel,0.085719
2,temp_media,0.083551
3,temp_max_media,0.066016
18,provincia_Santa Fe,0.054815
10,provincia_Corrientes,0.034476


In [7]:
conn = sqlite3.connect("agro.db")
b = pd.read_sql_query("""
    SELECT e.anio,
           SUM(e.superficie_sembrada_ha) AS area_soja,
           ec.precio_soja,
           r.retencion_soja,
           ec.tipo_cambio_prom
    FROM estimaciones e
    JOIN economia ec ON e.anio = ec.anio
    JOIN retenciones r ON e.anio = r.anio
    WHERE e.cultivo = 'soja total'
    GROUP BY e.anio
    ORDER BY e.anio
""", conn)
conn.close()

b["precio_neto"] = b["precio_soja"] * (1 - b["retencion_soja"] / 100)
b

,anio,area_soja,precio_soja,retencion_soja,tipo_cambio_prom,precio_neto
0,2000,10927330,211.83,0.0,NaN,211.83000
1,2001,11627961,195.83,0.0,NaN,195.83000
2,2002,12590244,212.67,23.5,NaN,162.69255
3,2003,14513106,264.00,23.5,NaN,201.96000
4,2004,14390630,306.50,23.5,NaN,234.47250
5,2005,15368083,274.69,23.5,NaN,210.13785
6,2006,16110038,268.65,23.5,NaN,205.51725
7,2007,16571885,383.10,27.5,NaN,277.74750
8,2008,17980895,521.87,35.0,NaN,339.21550
9,2009,18392752,423.62,35.0,NaN,275.35300


In [8]:
# 1) Correlación en NIVELES (ojo: la tendencia la ensucia)
corr_niveles = b["area_soja"].corr(b["precio_neto"])
print("Correlación área vs precio_neto (niveles):", round(corr_niveles, 3))

# 2) Cambios año a año (esto SACA la tendencia)
b["d_area"] = b["area_soja"].diff()
b["d_precio_neto"] = b["precio_neto"].diff()
corr_cambios = b["d_area"].corr(b["d_precio_neto"])
print("Correlación Δárea vs Δprecio_neto (cambios):", round(corr_cambios, 3))

b[["anio", "area_soja", "precio_neto", "d_area", "d_precio_neto"]]

Correlación área vs precio_neto (niveles): 0.54
Correlación Δárea vs Δprecio_neto (cambios): -0.223


,anio,area_soja,precio_neto,d_area,d_precio_neto
0,2000,10927330,211.83000,NaN,NaN
1,2001,11627961,195.83000,700631.0,-16.00000
2,2002,12590244,162.69255,962283.0,-33.13745
3,2003,14513106,201.96000,1922862.0,39.26745
4,2004,14390630,234.47250,-122476.0,32.51250
5,2005,15368083,210.13785,977453.0,-24.33465
6,2006,16110038,205.51725,741955.0,-4.62060
7,2007,16571885,277.74750,461847.0,72.23025
8,2008,17980895,339.21550,1409010.0,61.46800
9,2009,18392752,275.35300,411857.0,-63.86250


In [10]:
conn = sqlite3.connect("agro.db")
sub = pd.read_sql_query("""
    SELECT e.anio,
       SUM(CASE WHEN e.cultivo='soja total' THEN e.superficie_sembrada_ha ELSE 0 END) AS area_soja,
       SUM(CASE WHEN e.cultivo='maíz' THEN e.superficie_sembrada_ha ELSE 0 END) AS area_maiz,
       ec.precio_soja, ec.precio_maiz, r.retencion_soja, r.retencion_maiz
    FROM estimaciones e
    JOIN economia ec ON e.anio = ec.anio
    JOIN retenciones r ON e.anio = r.anio
    WHERE e.cultivo IN ('soja total','maíz')
    GROUP BY e.anio ORDER BY e.anio
""", conn)
conn.close()

sub["neto_soja"] = sub["precio_soja"] * (1 - sub["retencion_soja"]/100)
sub["neto_maiz"] = sub["precio_maiz"] * (1 - sub["retencion_maiz"]/100)
sub["ventaja_soja"] = sub["neto_soja"] / sub["neto_maiz"]
sub["share_soja"]   = sub["area_soja"] / (sub["area_soja"] + sub["area_maiz"])

print("Niveles:", round(sub["share_soja"].corr(sub["ventaja_soja"]), 3))
sub["d_share"]   = sub["share_soja"].diff()
sub["d_ventaja"] = sub["ventaja_soja"].diff()
print("Cambios:", round(sub["d_share"].corr(sub["d_ventaja"]), 3))

sub[["anio","share_soja","ventaja_soja","d_share","d_ventaja"]]

Niveles: 0.547
Cambios: 0.109


,anio,share_soja,ventaja_soja,d_share,d_ventaja
0,2000,0.757693,2.392748,NaN,NaN
1,2001,0.791577,2.184627,0.033884,-0.208121
2,2002,0.804457,2.048612,0.012880,-0.136016
3,2003,0.830483,2.395843,0.026026,0.347231
4,2004,0.808714,2.621562,-0.021769,0.225719
5,2005,0.828088,2.662129,0.019374,0.040568
6,2006,0.818256,2.108302,-0.009832,-0.553828
7,2007,0.796334,2.121376,-0.021921,0.013074
8,2008,0.837013,1.900410,0.040679,-0.220966
9,2009,0.833609,2.079580,-0.003404,0.179170
